In [3]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. Currently running on CPU.")

GPU is not available. Currently running on CPU.


In [4]:
!pip install -q sentence-transformers transformers datasets faiss-cpu pandas numpy scikit-learn matplotlib

In [5]:
import torch
import sentence_transformers
import transformers
import datasets
import faiss
import pandas as pd
import numpy as np
import sklearn
import matplotlib

print("All libraries imported successfully!")
print("-" * 40)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("CUDA available:", torch.cuda.is_available())


if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Running on CPU.")

All libraries imported successfully!
----------------------------------------
PyTorch: 2.11.0+cpu
Transformers: 5.21.0
Sentence Transformers: 5.7.0
CUDA available: False
No GPU detected. Running on CPU.


In [6]:
from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = "intfloat/multilingual-e5-small"
# choose GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading the Embedding Model")
model = SentenceTransformer(
    MODEL_NAME,
    device=device
)

print("\nModel loaded successfully!")
print("Model name:", MODEL_NAME)
print("Device:", device)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading the Embedding Model


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]


Model loaded successfully!
Model name: intfloat/multilingual-e5-small
Device: cpu
Embedding dimension: 384


/tmp/ipykernel_2025/350259951.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [7]:
# Sample English and Sanskrit texts
english_text = "A person should perform their duty without attachment to results."

sanskrit_text = "कर्मण्येवाधिकारस्ते मा फलेषु कदाचन।"

# E5 models use task prefixes for retrieval
english_input = "query: " + english_text
sanskrit_input = "passage: " + sanskrit_text

# Generate embeddings
embeddings = model.encode(
    [english_input, sanskrit_input],
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)
print("English embedding dimension:", len(embeddings[0]))
print("Sanskrit embedding dimension:", len(embeddings[1]))
print("Embeddings generated successfully!")

Embedding shape: (2, 384)
English embedding dimension: 384
Sanskrit embedding dimension: 384
Embeddings generated successfully!


In [8]:
!pip install -q huggingface_hub

In [9]:
from huggingface_hub import HfApi

api = HfApi()

results = api.list_datasets(
    search="Bhagavad Gita",
    limit=15
)

print("Potential Bhagavad Gita datasets:\n")

for dataset in results:
    print(dataset.id)

Potential Bhagavad Gita datasets:

OEvortex/Bhagavad_Gita
AmanKumar007/bhagavad-gita-shloka-dataset
SatyaSanatan/shrimad-bhagavad-gita-dataset-alpaca
Mercity/bhagavad_gita-embeddings
sweatSmile/Bhagavad-Gita-Vyasa-Edwin-Arnold
snskrt/Shrimad_Bhagavad_Gita
p2kalita/The-Bhagavad-Gita-with-a-TWIST
JDhruv14/Bhagavad-Gita_Dataset
abagade/bhagavad-gita-guidance-qa
dineshkarki/bhagavad-gita-nepali
nik-55/bhagavad-gita
dineshkarki/bhagavad-gita-nepali-sharegpt
JDhruv14/Bhagavad-Gita-QA
rahul7star/Bhagavad-Gita
Voider22/bhagavad-gita-verses-sanskrit-translations


In [10]:
from datasets import load_dataset

DATASET_NAME = "OEvortex/Bhagavad_Gita"

print("Loading dataset...")

dataset = load_dataset(
    DATASET_NAME,
    split="train"
)

print("\nDataset loaded successfully!")
print("Number of rows:", len(dataset))
print("Column names:", dataset.column_names)
print("\nDataset features:")
print(dataset.features)

Loading dataset...


README.md:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  288kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/700 [00:00<?, ? examples/s]


Dataset loaded successfully!
Number of rows: 700
Column names: ['S.No.', 'Title', 'Chapter', 'Verse', 'Sanskrit Anuvad', 'Hindi Anuvad', 'Enlgish Translation']

Dataset features:
{'S.No.': Value('int64'), 'Title': Value('string'), 'Chapter': Value('string'), 'Verse': Value('string'), 'Sanskrit Anuvad': Value('string'), 'Hindi Anuvad': Value('string'), 'Enlgish Translation': Value('string')}


In [11]:
import pandas as pd

df = dataset.to_pandas()

print("Dataset shape:", df.shape)

print("Columns", df.columns.to_list())

print("\nFirst five records:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nColumn data types:")
print(df.dtypes)

Dataset shape: (700, 7)
Columns ['S.No.', 'Title', 'Chapter', 'Verse', 'Sanskrit Anuvad', 'Hindi Anuvad', 'Enlgish Translation']

First five records:


,S.No.,Title,Chapter,Verse,Sanskrit Anuvad,Hindi Anuvad,Enlgish Translation
0,1,Arjuna's Vishada Yoga,Chapter 1,Verse 1.1,धृतराष्ट्र उवाच । धर्मक्षेत्रे कुरुक्षेत्रे सम...,धृतराष्ट्र बोले- हे संजय! धर्मभूमि कुरुक्षेत्र...,"Dhrtarashtra asked of Sanjaya: O SANJAYA, what..."
1,2,Arjuna's Vishada Yoga,Chapter 1,Verse 1.2,सञ्जय उवाच । दृष्ट्वा तु पाण्डवानीकं व्यूढं दु...,संजय बोले- उस समय राजा दुर्योधन ने व्यूहरचनायु...,Sanjaya explained: Now seeing that the army of...
2,3,Arjuna's Vishada Yoga,Chapter 1,Verse 1.3,पश्यैतां पाण्डुपुत्राणामाचार्य महतीं चमूम् । व...,हे आचार्य! आपके बुद्धिमान्‌ शिष्य द्रुपदपुत्र ...,"Behold O, Master, the mighty army of the sons ..."
3,4,Arjuna's Vishada Yoga,Chapter 1,Verse 1.4,अत्र शूरा महेष्वासा भीमार्जुनसमा युधि । युयुधा...,इस सेना में बड़े-बड़े धनुषों वाले तथा युद्ध मे...,"Present here are the mighty archers, peers or ..."
4,5,Arjuna's Vishada Yoga,Chapter 1,Verse 1.5,धृष्टकेतुश्चेकितानः काशिराजश्च वीर्यवान् । पुर...,"धृष्टकेतु, चेकिताना, और काशी के बहादुर राजा, प...","Dhrishtaketu, Chekitana, and the valiant king ..."



Missing values:
S.No.                  0
Title                  0
Chapter                0
Verse                  0
Sanskrit Anuvad        0
Hindi Anuvad           0
Enlgish Translation    0
dtype: int64

Column data types:
S.No.                   int64
Title                  object
Chapter                object
Verse                  object
Sanskrit Anuvad        object
Hindi Anuvad           object
Enlgish Translation    object
dtype: object


In [12]:
# checking the first complete record
print("\nFirst complete record:")
print(df.iloc[0].to_dict())


First complete record:
{'S.No.': 1, 'Title': "Arjuna's Vishada Yoga", 'Chapter': 'Chapter 1', 'Verse': 'Verse 1.1', 'Sanskrit Anuvad': 'धृतराष्ट्र उवाच । धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय ॥...', 'Hindi Anuvad': 'धृतराष्ट्र बोले- हे संजय! धर्मभूमि कुरुक्षेत्र में एकत्रित, युद्ध की इच्छावाले मेरे और पाण्डु के पुत्रों ने क्या किया?', 'Enlgish Translation': 'Dhrtarashtra asked of Sanjaya: O SANJAYA, what did my warrior sons and those of Pandu do when they were gathered at KURUKSHETRA, the field of religious activities? Tell me of those happenings.'}


In [13]:
print("Original dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nCompletely duplicated rows:", df.duplicated().sum())

print("\nDuplicated verse identifiers:", df["Verse"].duplicated().sum())

Original dataset shape: (700, 7)

Missing values:
S.No.                  0
Title                  0
Chapter                0
Verse                  0
Sanskrit Anuvad        0
Hindi Anuvad           0
Enlgish Translation    0
dtype: int64

Completely duplicated rows: 0

Duplicated verse identifiers: 0


In [14]:
import pandas as pd
import unicodedata

# Create a copy so the original dataset remains unchanged
clean_df = df.copy()

# Rename columns for easier access
clean_df = clean_df.rename(columns={
    "S.No.": "id",
    "Title": "chapter_title",
    "Chapter": "chapter",
    "Verse": "verse",
    "Sanskrit Anuvad": "sanskrit",
    "Hindi Anuvad": "hindi",
    "Enlgish Translation": "english"
})

# Normalize Unicode and remove surrounding whitespace
text_columns = ["chapter_title", "chapter", "verse",
                "sanskrit", "hindi", "english"]

for column in text_columns:
    clean_df[column] = clean_df[column].apply(
        lambda text: unicodedata.normalize("NFC", str(text)).strip()
        if pd.notna(text) else ""
    )

# Remove rows without Sanskrit or English text
clean_df = clean_df[
    (clean_df["sanskrit"] != "") &
    (clean_df["english"] != "")
].copy()

# Reset row indices
clean_df.reset_index(drop=True, inplace=True)

print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", clean_df.shape)

print("\nCleaned column names:")
print(clean_df.columns.tolist())

print("\nFirst three cleaned records:")
display(clean_df[[
    "id", "chapter", "verse", "sanskrit", "english"
]].head(3))

Original dataset shape: (700, 7)
Cleaned dataset shape: (700, 7)

Cleaned column names:
['id', 'chapter_title', 'chapter', 'verse', 'sanskrit', 'hindi', 'english']

First three cleaned records:


,id,chapter,verse,sanskrit,english
0,1,Chapter 1,Verse 1.1,धृतराष्ट्र उवाच । धर्मक्षेत्रे कुरुक्षेत्रे सम...,"Dhrtarashtra asked of Sanjaya: O SANJAYA, what..."
1,2,Chapter 1,Verse 1.2,सञ्जय उवाच । दृष्ट्वा तु पाण्डवानीकं व्यूढं दु...,Sanjaya explained: Now seeing that the army of...
2,3,Chapter 1,Verse 1.3,पश्यैतां पाण्डुपुत्राणामाचार्य महतीं चमूम् । व...,"Behold O, Master, the mighty army of the sons ..."


In [15]:
# Step 6.2: Create the passage collection

passages_df = clean_df[
    ["id", "chapter", "verse", "sanskrit", "english"]
].copy()

# Combine Sanskrit and English into a single searchable passage
passages_df["passage"] = (
    "Sanskrit: " + passages_df["sanskrit"].astype(str)
    + "\nEnglish: " + passages_df["english"].astype(str)
)

print("Total passages:", len(passages_df))

print("\nSample passages:")
display(passages_df[["id", "verse", "passage"]].head(3))

Total passages: 700

Sample passages:


,id,verse,passage
0,1,Verse 1.1,Sanskrit: धृतराष्ट्र उवाच । धर्मक्षेत्रे कुरुक...
1,2,Verse 1.2,Sanskrit: सञ्जय उवाच । दृष्ट्वा तु पाण्डवानीकं...
2,3,Verse 1.3,Sanskrit: पश्यैतां पाण्डुपुत्राणामाचार्य महतीं...


In [16]:
# Step 6.3: Create initial cross-lingual training pairs

training_df = clean_df[
    ["id", "chapter", "verse", "sanskrit", "english"]
].copy()

# English query -> relevant Sanskrit passage
training_df["query"] = training_df["english"].apply(
    lambda text: "query: " + text
)

training_df["positive"] = training_df["sanskrit"].apply(
    lambda text: "passage: " + text
)

print("Total aligned pairs:", len(training_df))

print("\nSample training pairs:")
display(training_df[["verse", "query", "positive"]].head(3))

Total aligned pairs: 700

Sample training pairs:


,verse,query,positive
0,Verse 1.1,query: Dhrtarashtra asked of Sanjaya: O SANJAY...,passage: धृतराष्ट्र उवाच । धर्मक्षेत्रे कुरुक्...
1,Verse 1.2,query: Sanjaya explained: Now seeing that the ...,passage: सञ्जय उवाच । दृष्ट्वा तु पाण्डवानीकं ...
2,Verse 1.3,"query: Behold O, Master, the mighty army of th...",passage: पश्यैतां पाण्डुपुत्राणामाचार्य महतीं ...


In [17]:
# Step 6.4: Validate the training pairs

print("Number of training pairs:", len(training_df))

print("\nMissing queries:", training_df["query"].isna().sum())
print("Missing positive passages:", training_df["positive"].isna().sum())

print("\nEmpty queries:", (training_df["query"].str.strip() == "query:").sum())
print(
    "Empty positive passages:",
    (training_df["positive"].str.strip() == "passage:").sum()
)

print("\nFirst training example:")
print("QUERY:", training_df.iloc[0]["query"])
print("\nPOSITIVE:", training_df.iloc[0]["positive"])

Number of training pairs: 700

Missing queries: 0
Missing positive passages: 0

Empty queries: 0
Empty positive passages: 0

First training example:
QUERY: query: Dhrtarashtra asked of Sanjaya: O SANJAYA, what did my warrior sons and those of Pandu do when they were gathered at KURUKSHETRA, the field of religious activities? Tell me of those happenings.

POSITIVE: passage: धृतराष्ट्र उवाच । धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय ॥...


In [18]:
from sklearn.model_selection import train_test_split
# Step 1: Separate the test dataset (20%)
train_val_df, test_df = train_test_split(
    clean_df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

# Step 2: Separate training and validation datasets
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.125,
    random_state=42,
    shuffle=True
)

# Step 3: Display dataset sizes
print("Total records:", len(clean_df))
print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Testing records:", len(test_df))

Total records: 700
Training records: 490
Validation records: 70
Testing records: 140


In [19]:
train_ids = set(train_df["id"])
val_ids = set(val_df["id"])
test_ids = set(test_df["id"])

print("Train and validation overlap:", len(train_ids & val_ids))
print("Train and test overlap:", len(train_ids & test_ids))
print("Validation and test overlap:", len(val_ids & test_ids))

Train and validation overlap: 0
Train and test overlap: 0
Validation and test overlap: 0


In [20]:
import pandas as pd

def prepare_pairs(dataframe):
    pairs = dataframe[["id", "chapter", "verse", "english", "sanskrit"]].copy()

    pairs["query"] = "query: " + pairs["english"].astype(str)
    pairs["positive"] = "passage: " + pairs["sanskrit"].astype(str)

    return pairs.reset_index(drop=True)


# Prepare each dataset separately
train_pairs = prepare_pairs(train_df)
val_pairs = prepare_pairs(val_df)
test_pairs = prepare_pairs(test_df)

print("Training pairs:", len(train_pairs))
print("Validation pairs:", len(val_pairs))
print("Testing pairs:", len(test_pairs))

print("\nExample training pair:")
print("English query:", train_pairs.iloc[0]["query"])
print("Sanskrit passage:", train_pairs.iloc[0]["positive"])

Training pairs: 490
Validation pairs: 70
Testing pairs: 140

Example training pair:
English query: query: Dear Lord Supreme, I do not understand Your mysterious ways. Reveal Yourself to me! Who are You in this terrible form? Be gracious to me Dear Lord, and explain to me the secret of Your reality, for I am confused.
Sanskrit passage: passage: आख्याहि मे को भवानुग्ररूपो नमोऽस्तु ते देववर प्रसीद । विज्ञातुमिच्छामि भवन्तमाद्यं न हि प्रजानामि...


In [21]:
for name, pairs in [
    ("Training", train_pairs),
    ("Validation", val_pairs),
    ("Testing", test_pairs)
]:
    print(f"\n{name} dataset")
    print("Missing queries:", pairs["query"].isna().sum())
    print("Missing passages:", pairs["positive"].isna().sum())
    print("Empty queries:", (pairs["query"].str.strip() == "query:").sum())
    print("Empty passages:", (pairs["positive"].str.strip() == "passage:").sum())


Training dataset
Missing queries: 0
Missing passages: 0
Empty queries: 0
Empty passages: 0

Validation dataset
Missing queries: 0
Missing passages: 0
Empty queries: 0
Empty passages: 0

Testing dataset
Missing queries: 0
Missing passages: 0
Empty queries: 0
Empty passages: 0


In [22]:
# Check the length of the text in characters

print("English text length:")
print("Maximum:", train_pairs["english"].str.len().max())
print("Average:", round(train_pairs["english"].str.len().mean(), 2))

print("\nSanskrit text length:")
print("Maximum:", train_pairs["sanskrit"].str.len().max())
print("Average:", round(train_pairs["sanskrit"].str.len().mean(), 2))

# Inspect the longest English example
longest_idx = train_pairs["english"].str.len().idxmax()

print("\nLongest English example:")
print(train_pairs.loc[longest_idx, "english"])

print("\nCorresponding Sanskrit passage:")
print(train_pairs.loc[longest_idx, "sanskrit"])

English text length:
Maximum: 643
Average: 231.83

Sanskrit text length:
Maximum: 107
Average: 92.43

Longest English example:
‘I am wealthy and come from a very well-to-do and quite noble family. Who else in the world can compare to my greatness? I shall sacrifice during religious rituals (whether I believe in them or not).I shall give to various charities (to be celebrated as a great supporter of these charities) I shall thoroughly enjoy life as I please.’This, O Arjuna, is what people who are in a deluded,confused and corrupted state say.These peole who are confused by many foul thoughts and evil influences, entangled in the net of unending darkness and obsessive desires for material gains, they are blackened souls who fall down into a foul hell in the end.

Corresponding Sanskrit passage:
आढ्योऽभिजनवानस्मि कोऽन्योऽस्ति सदृशो मया । यक्ष्ये दास्यामि मोदिष्य इत्यज्ञानविमोहिताः ॥ १६.१५ ॥


In [23]:
# Get the tokenizer from our existing embedding model
tokenizer = model.tokenizer

# Check token lengths for the training data
train_pairs["english_tokens"] = train_pairs["english"].apply(
    lambda text: len(tokenizer.encode(text, add_special_tokens=True))
)

train_pairs["sanskrit_tokens"] = train_pairs["sanskrit"].apply(
    lambda text: len(tokenizer.encode(text, add_special_tokens=True))
)

print("English token statistics:")
print("Maximum:", train_pairs["english_tokens"].max())
print("Average:", round(train_pairs["english_tokens"].mean(), 2))

print("\nSanskrit token statistics:")
print("Maximum:", train_pairs["sanskrit_tokens"].max())
print("Average:", round(train_pairs["sanskrit_tokens"].mean(), 2))

print("\nModel maximum sequence length:", model.max_seq_length)

English token statistics:
Maximum: 169
Average: 60.79

Sanskrit token statistics:
Maximum: 48
Average: 37.72

Model maximum sequence length: 512


In [24]:
# Create a Sanskrit-only retrieval corpus
corpus_df = clean_df[["id", "chapter", "verse", "sanskrit"]].copy()

# Prepare English queries from the held-out test set
evaluation_df = test_df[["id", "chapter", "verse", "english", "sanskrit"]].copy()

print("Number of Sanskrit passages:", len(corpus_df))
print("Number of evaluation queries:", len(evaluation_df))

print("\nExample Sanskrit passage:")
print(corpus_df.iloc[0]["sanskrit"])

print("\nExample English query:")
print(evaluation_df.iloc[0]["english"])

print("\nCorrect Sanskrit verse for this query:")
print(evaluation_df.iloc[0]["sanskrit"])

Number of Sanskrit passages: 700
Number of evaluation queries: 140

Example Sanskrit passage:
धृतराष्ट्र उवाच । धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय ॥...

Example English query:
The Blessed Lord advised: Therefore, O Arjuna, restrain the senses first and control your sinful desires, the enemy of Gyan or knowledge.

Correct Sanskrit verse for this query:
तस्मात्त्वमिन्द्रियाण्यादौ नियम्य भरतर्षभ । पाप्मानं प्रजहि ह्येनं ज्ञानविज्ञाननाशनम् ॥ ३.४१ ॥


In [25]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. Encode all Sanskrit passages
corpus_texts = [
    "passage: " + text
    for text in corpus_df["sanskrit"].astype(str)
]

corpus_embeddings = model.encode(
    corpus_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

# 2. Encode the English evaluation queries
query_texts = [
    "query: " + text
    for text in evaluation_df["english"].astype(str)
]

query_embeddings = model.encode(
    query_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

# 3. Calculate cosine similarity between queries and passages
similarity_matrix = query_embeddings @ corpus_embeddings.T

# 4. Retrieve the five most similar Sanskrit passages per query
top_k = 5
top_indices = np.argsort(-similarity_matrix, axis=1)[:, :top_k]

# 5. Calculate Recall@5
corpus_ids = corpus_df["id"].to_numpy()
correct_ids = evaluation_df["id"].to_numpy()

hits = []

for i in range(len(evaluation_df)):
    retrieved_ids = corpus_ids[top_indices[i]]
    hits.append(correct_ids[i] in retrieved_ids)

recall_at_5 = np.mean(hits)

print("BASE MODEL EVALUATION")
print("---------------------")
print("Number of queries:", len(evaluation_df))
print("Number of passages:", len(corpus_df))
print("Recall@5:", round(float(recall_at_5), 4))
print("Recall@5 (%):", round(float(recall_at_5) * 100, 2))

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

BASE MODEL EVALUATION
---------------------
Number of queries: 140
Number of passages: 700
Recall@5: 0.2429
Recall@5 (%): 24.29


In [26]:
# Inspect the first five evaluation queries

for i in range(5):
    query = evaluation_df.iloc[i]["english"]
    correct_verse = evaluation_df.iloc[i]["verse"]
    correct_sanskrit = evaluation_df.iloc[i]["sanskrit"]

    print("=" * 80)
    print(f"QUERY {i + 1}")
    print("\nEnglish query:")
    print(query)

    print("\nExpected verse:", correct_verse)
    print("Expected Sanskrit:")
    print(correct_sanskrit)

    print("\nTop 5 retrieved passages:")

    for rank, idx in enumerate(top_indices[i], start=1):
        retrieved = corpus_df.iloc[idx]
        score = similarity_matrix[i, idx]

        print(f"\nRank {rank}")
        print("Verse:", retrieved["verse"])
        print("Similarity:", round(float(score), 4))
        print("Sanskrit:", retrieved["sanskrit"])

        if retrieved["id"] == evaluation_df.iloc[i]["id"]:
            print("CORRECT MATCH")

QUERY 1

English query:
The Blessed Lord advised: Therefore, O Arjuna, restrain the senses first and control your sinful desires, the enemy of Gyan or knowledge.

Expected verse: Verse 3.41
Expected Sanskrit:
तस्मात्त्वमिन्द्रियाण्यादौ नियम्य भरतर्षभ । पाप्मानं प्रजहि ह्येनं ज्ञानविज्ञाननाशनम् ॥ ३.४१ ॥

Top 5 retrieved passages:

Rank 1
Verse: Verse 11.51
Similarity: 0.9004
Sanskrit: अर्जुन उवाच । दृष्ट्वेदं मानुषं रूपं तव सौम्यं जनार्दन इदानीमस्मि संवृत्तः सचेताः प्रकृतिं गतः ॥ १...

Rank 2
Verse: Verse 2.11
Similarity: 0.895
Sanskrit: श्रीभगवानुवाच । अशोच्यानन्वशोचस्त्वं प्रज्ञावादांश्च भाषसे गतासूनगतासूंश्च नानुशोचन्ति पण्डिताः ॥ ...

Rank 3
Verse: Verse 1.28
Similarity: 0.8948
Sanskrit: कृपया परयाविष्टो विषीदन्निदमब्रवीत् | अर्जुन उवाच | दृष्ट्वेमं स्वजनं कृष्ण युयुत्सुं समुपस्थितम्

Rank 4
Verse: Verse 9.1
Similarity: 0.8938
Sanskrit: श्रीभगवानुवाच । इदं तु ते गुह्यतमं प्रवक्ष्याम्यनसूयवे ज्ञानं विज्ञानसहितं यज्ज्ञात्वा मोक्ष्यसेऽश...

Rank 5
Verse: Verse 2.39
Similarity: 0.8897
S

In [27]:
# Inspect the first five evaluation queries

for i in range(5):
    query = evaluation_df.iloc[i]["english"]
    correct_verse = evaluation_df.iloc[i]["verse"]
    correct_sanskrit = evaluation_df.iloc[i]["sanskrit"]

    print("=" * 80)
    print(f"QUERY {i + 1}")
    print("\nEnglish query:")
    print(query)

    print("\nExpected verse:", correct_verse)
    print("Expected Sanskrit:")
    print(correct_sanskrit)

    print("\nTop 5 retrieved passages:")

    for rank, idx in enumerate(top_indices[i], start=1):
        retrieved = corpus_df.iloc[idx]
        score = similarity_matrix[i, idx]

        print(f"\nRank {rank}")
        print("Verse:", retrieved["verse"])
        print("Similarity:", round(float(score), 4))
        print("Sanskrit:", retrieved["sanskrit"])

        if retrieved["id"] == evaluation_df.iloc[i]["id"]:
            print("CORRECT MATCH")

QUERY 1

English query:
The Blessed Lord advised: Therefore, O Arjuna, restrain the senses first and control your sinful desires, the enemy of Gyan or knowledge.

Expected verse: Verse 3.41
Expected Sanskrit:
तस्मात्त्वमिन्द्रियाण्यादौ नियम्य भरतर्षभ । पाप्मानं प्रजहि ह्येनं ज्ञानविज्ञाननाशनम् ॥ ३.४१ ॥

Top 5 retrieved passages:

Rank 1
Verse: Verse 11.51
Similarity: 0.9004
Sanskrit: अर्जुन उवाच । दृष्ट्वेदं मानुषं रूपं तव सौम्यं जनार्दन इदानीमस्मि संवृत्तः सचेताः प्रकृतिं गतः ॥ १...

Rank 2
Verse: Verse 2.11
Similarity: 0.895
Sanskrit: श्रीभगवानुवाच । अशोच्यानन्वशोचस्त्वं प्रज्ञावादांश्च भाषसे गतासूनगतासूंश्च नानुशोचन्ति पण्डिताः ॥ ...

Rank 3
Verse: Verse 1.28
Similarity: 0.8948
Sanskrit: कृपया परयाविष्टो विषीदन्निदमब्रवीत् | अर्जुन उवाच | दृष्ट्वेमं स्वजनं कृष्ण युयुत्सुं समुपस्थितम्

Rank 4
Verse: Verse 9.1
Similarity: 0.8938
Sanskrit: श्रीभगवानुवाच । इदं तु ते गुह्यतमं प्रवक्ष्याम्यनसूयवे ज्ञानं विज्ञानसहितं यज्ज्ञात्वा मोक्ष्यसेऽश...

Rank 5
Verse: Verse 2.39
Similarity: 0.8897
S

In [28]:
from sentence_transformers import InputExample

train_examples = [
    InputExample(
        texts=[row["query"], row["positive"]]
    )
    for _, row in train_pairs.iterrows()
]

print("Number of training examples:", len(train_examples))

print("\nFirst training example:")
print("English query:", train_examples[0].texts[0])
print("Positive Sanskrit passage:", train_examples[0].texts[1])

Number of training examples: 490

First training example:
English query: query: Dear Lord Supreme, I do not understand Your mysterious ways. Reveal Yourself to me! Who are You in this terrible form? Be gracious to me Dear Lord, and explain to me the secret of Your reality, for I am confused.
Positive Sanskrit passage: passage: आख्याहि मे को भवानुग्ररूपो नमोऽस्तु ते देववर प्रसीद । विज्ञातुमिच्छामि भवन्तमाद्यं न हि प्रजानामि...


In [29]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected by PyTorch.")

PyTorch version: 2.11.0+cpu
CUDA available: False
CUDA version: None
No GPU detected by PyTorch.


In [30]:
import sentence_transformers
import torch

print("Sentence Transformers version:", sentence_transformers.__version__)
print("PyTorch version:", torch.__version__)
print("Model maximum sequence length:", model.max_seq_length)
print("Training examples:", len(train_examples))
print("Device:", device)

Sentence Transformers version: 5.7.0
PyTorch version: 2.11.0+cpu
Model maximum sequence length: 512
Training examples: 490
Device: cpu


In [34]:
from datasets import Dataset

# Convert training pairs
train_dataset = Dataset.from_dict({
    "anchor": train_pairs["query"].tolist(),
    "positive": train_pairs["positive"].tolist()
})

# Convert validation pairs
val_dataset = Dataset.from_dict({
    "anchor": val_pairs["query"].tolist(),
    "positive": val_pairs["positive"].tolist()
})

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

print("\nFirst training example:")
print("Query:", train_dataset[0]["anchor"])
print("Positive:", train_dataset[0]["positive"])

print("\nDataset preparation completed!")


Training dataset: 490
Validation dataset: 70

First training example:
Query: query: Dear Lord Supreme, I do not understand Your mysterious ways. Reveal Yourself to me! Who are You in this terrible form? Be gracious to me Dear Lord, and explain to me the secret of Your reality, for I am confused.
Positive: passage: आख्याहि मे को भवानुग्ररूपो नमोऽस्तु ते देववर प्रसीद । विज्ञातुमिच्छामि भवन्तमाद्यं न हि प्रजानामि...

Dataset preparation completed!


In [38]:
from sentence_transformers import losses

# Recreate the contrastive-learning loss
train_loss = losses.MultipleNegativesRankingLoss(model)

print("Training loss recreated successfully!")
print("Loss function:", type(train_loss).__name__)
print("Model:", MODEL_NAME)
print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

Training loss recreated successfully!
Loss function: MultipleNegativesRankingLoss
Model: intfloat/multilingual-e5-small
Training examples: 490
Validation examples: 70


/tmp/ipykernel_2025/3903730185.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses


In [39]:
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

training_args = SentenceTransformerTrainingArguments(
    output_dir="./results/sanskrit_english_e5",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    warmup_steps=0.1,  # 10% warmup, expressed as a fraction
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=10,
    report_to="none",
    fp16=False,
    bf16=False,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=train_loss,
)

print("Updated training configuration prepared!")
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Learning rate:", training_args.learning_rate)
print("Warmup setting:", training_args.warmup_steps)
print("Device:", device)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Updated training configuration prepared!
Epochs: 2
Batch size: 8
Learning rate: 2e-05
Warmup setting: 0.1
Device: cpu


In [40]:
# Start fine-tuning
train_result = trainer.train()

print("\nFine-tuning completed successfully!")
print("Training metrics:")
print(train_result.metrics)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,1.707236
20,1.253050
30,0.814690
40,0.577308
50,0.582856
60,0.565648
70,0.402294
80,0.320962
90,0.371760
100,0.365814


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuning completed successfully!
Training metrics:
{'train_runtime': 673.6215, 'train_samples_per_second': 1.455, 'train_steps_per_second': 0.184, 'total_flos': 0.0, 'train_loss': 0.6111081213720383, 'epoch': 2.0}


In [41]:
import os

print("Trainer available:", "trainer" in globals())
print("Model available:", "model" in globals())

print("\nCurrent directory:", os.getcwd())
print("Models directory exists:", os.path.exists("./models"))

if os.path.exists("./models"):
    print("Models directory contents:", os.listdir("./models"))

Trainer available: True
Model available: True

Current directory: /content
Models directory exists: False


In [42]:
import os

model_save_path = "/content/models/sanskrit_english_e5_finetuned"

os.makedirs(model_save_path, exist_ok=True)

trainer.save_model(model_save_path)

print("Fine-tuned model saved successfully!")
print("Model location:", model_save_path)
print("Files saved:", os.listdir(model_save_path))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved successfully!
Model location: /content/models/sanskrit_english_e5_finetuned
Files saved: ['config_sentence_transformers.json', '1_Pooling', 'README.md', 'training_args.bin', '2_Normalize', 'config.json', 'tokenizer.json', 'modules.json', 'model.safetensors', 'sentence_bert_config.json', 'tokenizer_config.json']


In [43]:
# Check whether the current trainer completed training
print("Training steps completed:", trainer.state.global_step)
print("Training epoch:", trainer.state.epoch)

if trainer.state.global_step >= 124 and trainer.state.epoch is not None and trainer.state.epoch >= 2:
    print("\nTraining state indicates that both epochs completed.")
else:
    print("\nWarning: The current trainer does not confirm completion of both epochs.")
    print("We should verify the saved weights before evaluating the model.")

Training steps completed: 124
Training epoch: 2.0

Training state indicates that both epochs completed.


In [44]:
from sentence_transformers import SentenceTransformer
import os

model_save_path = "/content/models/sanskrit_english_e5_finetuned"

# Verify that the model file exists
assert os.path.exists(
    os.path.join(model_save_path, "model.safetensors")
), "Model weights file not found!"

# Load the saved model
fine_tuned_model = SentenceTransformer(model_save_path)

print("Fine-tuned model loaded successfully!")
print("Model path:", model_save_path)
print("Embedding dimension:", fine_tuned_model.get_sentence_embedding_dimension())
print("Maximum sequence length:", fine_tuned_model.max_seq_length)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fine-tuned model loaded successfully!
Model path: /content/models/sanskrit_english_e5_finetuned
Embedding dimension: 384
Maximum sequence length: 512


/tmp/ipykernel_2025/2195494495.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", fine_tuned_model.get_sentence_embedding_dimension())


In [45]:
import numpy as np

# Prepare the Sanskrit retrieval corpus
corpus_df = clean_df[["id", "chapter", "verse", "sanskrit"]].copy()

corpus_texts = [
    "passage: " + text
    for text in corpus_df["sanskrit"].astype(str)
]

# Generate embeddings using the fine-tuned model
fine_tuned_corpus_embeddings = fine_tuned_model.encode(
    corpus_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Sanskrit corpus embeddings generated successfully!")
print("Number of verses:", len(corpus_df))
print("Embedding shape:", fine_tuned_corpus_embeddings.shape)
print("Normalized:", np.allclose(
    np.linalg.norm(fine_tuned_corpus_embeddings, axis=1),
    1.0,
    atol=1e-4
))

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

Sanskrit corpus embeddings generated successfully!
Number of verses: 700
Embedding shape: (700, 384)
Normalized: True


In [46]:
import numpy as np
import pandas as pd

# Prepare English test queries
evaluation_df = test_df[
    ["id", "chapter", "verse", "english", "sanskrit"]
].copy().reset_index(drop=True)

query_texts = [
    "query: " + text
    for text in evaluation_df["english"].astype(str)
]

# Generate query embeddings using the fine-tuned model
fine_tuned_query_embeddings = fine_tuned_model.encode(
    query_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

# Calculate cosine similarity scores
similarity_matrix_ft = (
    fine_tuned_query_embeddings @ fine_tuned_corpus_embeddings.T
)

# Retrieve the top 5 Sanskrit verses for each query
top_k = 5

top_indices_ft = np.argsort(
    -similarity_matrix_ft,
    axis=1
)[:, :top_k]

corpus_ids = corpus_df["id"].to_numpy()
correct_ids = evaluation_df["id"].to_numpy()

# Calculate Recall@5
hits_ft = np.array([
    correct_ids[i] in corpus_ids[top_indices_ft[i]]
    for i in range(len(evaluation_df))
])

recall_at_5_ft = hits_ft.mean()

print("Fine-tuned model retrieval completed!")
print("Test queries:", len(evaluation_df))
print("Sanskrit verses searched:", len(corpus_df))
print("Recall@5:", round(recall_at_5_ft, 4))
print("Recall@5 (%):", round(recall_at_5_ft * 100, 2))
print("Queries with correct verse in top 5:", int(hits_ft.sum()))

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Fine-tuned model retrieval completed!
Test queries: 140
Sanskrit verses searched: 700
Recall@5: 0.5857
Recall@5 (%): 58.57
Queries with correct verse in top 5: 82


In [47]:
import numpy as np

# Rank all Sanskrit verses for each English query
ranked_indices = np.argsort(-similarity_matrix_ft, axis=1)

corpus_ids = corpus_df["id"].to_numpy()
correct_ids = evaluation_df["id"].to_numpy()

# Find the rank of the correct verse for each query
correct_ranks = []

for i, correct_id in enumerate(correct_ids):
    ranked_ids = corpus_ids[ranked_indices[i]]

    matching_positions = np.where(ranked_ids == correct_id)[0]

    if len(matching_positions) > 0:
        # Convert zero-based position into a one-based rank
        rank = int(matching_positions[0]) + 1
    else:
        rank = None

    correct_ranks.append(rank)

# Calculate retrieval metrics
recall_at_1 = np.mean([
    rank is not None and rank <= 1 for rank in correct_ranks
])

recall_at_3 = np.mean([
    rank is not None and rank <= 3 for rank in correct_ranks
])

recall_at_5 = np.mean([
    rank is not None and rank <= 5 for rank in correct_ranks
])

reciprocal_ranks = [
    1 / rank if rank is not None else 0
    for rank in correct_ranks
]

mrr = np.mean(reciprocal_ranks)

# Display results
print("Fine-Tuned Model Evaluation")
print("-" * 35)
print(f"Test queries: {len(evaluation_df)}")
print(f"Recall@1: {recall_at_1:.4f} ({recall_at_1 * 100:.2f}%)")
print(f"Recall@3: {recall_at_3:.4f} ({recall_at_3 * 100:.2f}%)")
print(f"Recall@5: {recall_at_5:.4f} ({recall_at_5 * 100:.2f}%)")
print(f"MRR:      {mrr:.4f}")

Fine-Tuned Model Evaluation
-----------------------------------
Test queries: 140
Recall@1: 0.3714 (37.14%)
Recall@3: 0.5286 (52.86%)
Recall@5: 0.5857 (58.57%)
MRR:      0.4811


In [48]:
# Inspect 5 test queries and their top 5 retrieved Sanskrit verses

num_examples = 5
top_k = 5

for i in range(num_examples):
    print("=" * 90)
    print(f"TEST QUERY {i + 1}")
    print(f"English question/translation:\n{evaluation_df.iloc[i]['english']}")

    print(
        f"\nCorrect verse ID: {evaluation_df.iloc[i]['id']}"
        f" | Chapter: {evaluation_df.iloc[i]['chapter']}"
        f" | Verse: {evaluation_df.iloc[i]['verse']}"
    )

    print("\nTop retrieved verses:")

    for rank, corpus_idx in enumerate(top_indices_ft[i], start=1):
        retrieved = corpus_df.iloc[corpus_idx]
        similarity = similarity_matrix_ft[i, corpus_idx]

        is_correct = retrieved["id"] == evaluation_df.iloc[i]["id"]

        print(f"\nRank: {rank}")
        print(f"Verse ID: {retrieved['id']}")
        print(f"Chapter: {retrieved['chapter']}, Verse: {retrieved['verse']}")
        print(f"Similarity: {similarity:.4f}")
        print(f"Exact match: {'YES' if is_correct else 'NO'}")
        print(f"Sanskrit: {retrieved['sanskrit']}")

TEST QUERY 1
English question/translation:
The Blessed Lord advised: Therefore, O Arjuna, restrain the senses first and control your sinful desires, the enemy of Gyan or knowledge.

Correct verse ID: 159 | Chapter: Chapter 3 | Verse: Verse 3.41

Top retrieved verses:

Rank: 1
Verse ID: 57
Chapter: Chapter 2, Verse: Verse 2.11
Similarity: 0.4597
Exact match: NO
Sanskrit: श्रीभगवानुवाच । अशोच्यानन्वशोचस्त्वं प्रज्ञावादांश्च भाषसे गतासूनगतासूंश्च नानुशोचन्ति पण्डिताः ॥ ...

Rank: 2
Verse ID: 267
Chapter: Chapter 6, Verse: Verse 6.35
Similarity: 0.4002
Exact match: NO
Sanskrit: श्रीभगवानुवाच । असंशयं महाबाहो मनो दुर्निग्रहं चलम् अभ्यासेन तु कौन्तेय वैराग्येण च गृह्यते ॥ ६.३५...

Rank: 3
Verse ID: 338
Chapter: Chapter 9, Verse: Verse 9.1
Similarity: 0.3790
Exact match: NO
Sanskrit: श्रीभगवानुवाच । इदं तु ते गुह्यतमं प्रवक्ष्याम्यनसूयवे ज्ञानं विज्ञानसहितं यज्ज्ञात्वा मोक्ष्यसेऽश...

Rank: 4
Verse ID: 241
Chapter: Chapter 6, Verse: Verse 6.9
Similarity: 0.3741
Exact match: NO
Sanskrit: सुहृन

In [49]:
import numpy as np

# Identify test queries where the correct verse is not in the top 5
failed_indices = np.where(~hits_ft)[0]

print("Total failed queries:", len(failed_indices))
print("Total test queries:", len(evaluation_df))
print("Overall Recall@5:", f"{hits_ft.mean() * 100:.2f}%")

# Select up to 5 failed queries reproducibly
rng = np.random.default_rng(42)
sample_size = min(5, len(failed_indices))
sampled_failures = rng.choice(
    failed_indices,
    size=sample_size,
    replace=False
)

for i in sampled_failures:
    query = evaluation_df.iloc[i]

    print("\n" + "=" * 90)
    print(f"Test index: {i}")
    print(f"English translation: {query['english']}")
    print(
        f"Correct verse: Chapter {query['chapter']}, "
        f"{query['verse']} | ID: {query['id']}"
    )
    print(f"Correct Sanskrit: {query['sanskrit']}")

    print("\nTop 3 retrieved results:")

    for rank, corpus_idx in enumerate(top_indices_ft[i][:3], start=1):
        result = corpus_df.iloc[corpus_idx]
        similarity = similarity_matrix_ft[i, corpus_idx]

        print(
            f"\nRank {rank} | "
            f"Chapter {result['chapter']} | "
            f"{result['verse']} | "
            f"ID: {result['id']} | "
            f"Similarity: {similarity:.4f}"
        )
        print(f"Sanskrit: {result['sanskrit']}")

Total failed queries: 58
Total test queries: 140
Overall Recall@5: 58.57%

Test index: 105
English translation: Arjuna, know that I am the seed of all things that are, and that no being that moves or moves not can ever be without Me.
Correct verse: Chapter Chapter 10, Verse 10.39 | ID: 410
Correct Sanskrit: यच्चापि सर्वभूतानां बीजं तदहमर्जुन । न तदस्ति विना यत्स्यान्मया भूतं चराचरम् ॥ १०.३९ ॥

Top 3 retrieved results:

Rank 1 | Chapter Chapter 6 | Verse 6.30 | ID: 262 | Similarity: 0.4683
Sanskrit: यो मां पश्यति सर्वत्र सर्वं च मयि पश्यति । तस्याहं न प्रणश्यामि स च मे न प्रणश्यति ॥ ६.३० ॥

Rank 2 | Chapter Chapter 9 | Verse 9.4 | ID: 341 | Similarity: 0.4653
Sanskrit: मया ततमिदं सर्वं जगदव्यक्तमूर्तिना । मत्स्थानि सर्वभूतानि न चाहं तेष्ववस्थितः ॥ ९.४ ॥

Rank 3 | Chapter Chapter 7 | Verse 7.7 | ID: 286 | Similarity: 0.4618
Sanskrit: मत्तः परतरं नान्यत्किंचिदस्ति धनंजय । मयि सर्वमिदं प्रोतं सूत्रे मणिगणा इव ॥ ७.७ ॥

Test index: 58
English translation: Once one has completely given up all

In [51]:
import pandas as pd

metrics_df = pd.DataFrame({
    "Metric": [
        "Recall@1",
        "Recall@3",
        "Recall@5",
        "MRR"
    ],
    "Original Model": [
        None,  # Baseline Recall@1 not calculated yet
        None,  # Baseline Recall@3 not calculated yet
        0.2429,
        None   # Baseline MRR not calculated yet
    ],
    "Fine-Tuned Model": [
        recall_at_1,
        recall_at_3,
        recall_at_5,
        mrr
    ]
})

metrics_df.to_csv("evaluation_results.csv", index=False)

print("Evaluation results saved successfully!")
display(metrics_df)

Evaluation results saved successfully!


,Metric,Original Model,Fine-Tuned Model
0,Recall@1,NaN,0.371429
1,Recall@3,NaN,0.528571
2,Recall@5,0.2429,0.585714
3,MRR,NaN,0.481077


In [52]:
import os

model_path = "/content/models/sanskrit_english_e5_finetuned"

if os.path.exists(os.path.join(model_path, "modules.json")):
    print("Fine-tuned model directory exists.")
    print("Files:")
    for filename in os.listdir(model_path):
        print("-", filename)
else:
    print("Model directory or modules.json not found.")

Fine-tuned model directory exists.
Files:
- config_sentence_transformers.json
- 1_Pooling
- README.md
- training_args.bin
- 2_Normalize
- config.json
- tokenizer.json
- modules.json
- model.safetensors
- sentence_bert_config.json
- tokenizer_config.json


In [53]:
import shutil
import os

model_path = "/content/models/sanskrit_english_e5_finetuned"
zip_path = "/content/sanskrit_english_e5_finetuned"

if os.path.exists(model_path):
    shutil.make_archive(zip_path, "zip", model_path)x
    print("Model packaged successfully!")
    print("ZIP file:", zip_path + ".zip")
    print(
        "File size:",
        round(os.path.getsize(zip_path + ".zip") / (1024 * 1024), 2),
        "MB"
    )
else:
    print("Model directory not found. Please check the path.")

Model packaged successfully!
ZIP file: /content/sanskrit_english_e5_finetuned.zip
File size: 299.55 MB


In [54]:
from google.colab import files

files.download("/content/sanskrit_english_e5_finetuned.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [55]:
import pandas as pd

metrics_df = pd.DataFrame({
    "Metric": ["Recall@1", "Recall@3", "Recall@5", "MRR"],
    "Baseline Model": [None, None, 0.2429, None],
    "Fine-Tuned Model": [
        recall_at_1,
        recall_at_3,
        recall_at_5,
        mrr
    ]
})

metrics_df.to_csv("/content/evaluation_results.csv", index=False)

print("Evaluation results saved!")
display(metrics_df)

Evaluation results saved!


,Metric,Baseline Model,Fine-Tuned Model
0,Recall@1,NaN,0.371429
1,Recall@3,NaN,0.528571
2,Recall@5,0.2429,0.585714
3,MRR,NaN,0.481077


In [56]:
from google.colab import files

files.download("/content/evaluation_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>